# Update MSCI Country Returns

Downloads monthly MSCI country index prices, computes returns, and appends new data to `country_returns.csv`.

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import MSCI

## Setup

Initialize the MSCI downloader and define the country index codes.

In [ ]:
msci = MSCI.MSCI(update_codes=True)

In [ ]:
msci_codes = {
    "Argentina": 903200,
    "Australia": 903600,
    "Austria": 904000,
    "Belgium": 905600,
    "Brazil": 907600,
    "Canada": 912400,
    "China": 302400,
    "Denmark": 920800,
    "Finland": 924600,
    "France": 925000,
    "Germany": 928000,
    "Greece": 930000,
    "Hong Kong": 934400,
    "India": 935600,
    "Indonesia": 105767,
    "Ireland": 937200,
    "Israel": 300400,
    "Italy": 938000,
    "Japan": 939200,
    "Korea": 941000,
    "Mexico": 848400,
    "Netherlands": 952800,
    "New Zealand": 955400,
    "Norway": 957800,
    "Philippines": 860800,
    "Portugal": 962000,
    "Singapore": 998100,
    "Spain": 972400,
    "Sweden": 975200,
    "Switzerland": 975600,
    "Taiwan": 915800,
    "Thailand": 105769,
    "Turkey": 979200,
    "United Kingdom": 982600,
    "USA": 984000
}

## Load existing data and determine date range

In [ ]:
rets_old = pd.read_csv('country_returns.csv', index_col=0, parse_dates=True)

# Start one month before last date (for overlap), end at today
start_date = (rets_old.index[-1] - timedelta(days=45)).strftime('%Y%m%d')
end_date = date.today().strftime('%Y%m%d')

print(f'Existing data ends: {rets_old.index[-1].date()}')
print(f'Downloading from:   {start_date} to {end_date}')

## Download new MSCI index prices

In [ ]:
data = pd.DataFrame({
    country: msci.get_index(country.upper(), start_date, end_date, freq='m')
    for country in msci_codes
})

rets = data.pct_change().dropna(how='all')
rets.index.name = 'Date'

print(f'Downloaded {len(rets)} months of new returns')
rets.tail()

## Merge with existing data

In [ ]:
rets_new = pd.concat([rets_old, rets]).sort_index()

# Keep first observation for each date (existing data takes priority)
rets_new = rets_new[~rets_new.index.duplicated(keep='first')]

## Validate

In [ ]:
assert rets_new.index.is_monotonic_increasing, 'Dates are not sorted'
assert rets_new.index.is_unique, 'Duplicate dates found'
assert rets_new.shape[1] == rets_old.shape[1], 'Column count changed'
assert np.all(rets_old.columns == rets_new.columns), 'Column names changed'
assert rets_new.shape[0] >= rets_old.shape[0], 'Lost rows'

n_new = rets_new.shape[0] - rets_old.shape[0]
print(f'Added {n_new} new month(s) of data')
print(f'Total: {rets_new.shape[0]} months, {rets_new.shape[1]} countries')
print(f'Date range: {rets_new.index[0].date()} to {rets_new.index[-1].date()}')

## Save

In [ ]:
rets_new.round(5).to_csv('country_returns.csv', index=True)